In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("whizzkid/fei-face-data")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import os, glob, re, random
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
# from torchvision import transforms, models
# from torchvision.utils import make_grid
from PIL import Image
from pathlib import Path
from collections import defaultdict

from scipy import stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
# --- Глобальные константы ---
DATA_ROOT = path
OUTPUT_PATH = "data/features/fei_features_min.npz"
ENCODER_PTH = "encoder.pth"
BATCH_SIZE = 128
OWN_COUNT = 20
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

# VAE

### Определение класса FEIFaceDataset

In [ ]:
import os
import re
from torch.utils.data import Dataset
import glob
from PIL import Image

class FEIFaceDataset(Dataset):
    """
    Возвращает изображение и метку (ID человека: 0–199)
    """
    def __init__(self, root, transform=None):
        self.transform = transform
        parts = [os.path.join(root, f"originalimages_part{i}") for i in range(1, 5)]
        self.files = []
        self.labels = []

        for p in parts:
            if os.path.isdir(p):
                jpg_files = sorted(glob.glob(os.path.join(p, "*.jpg")))
                for path in jpg_files:
                    fname = os.path.basename(path)
                    match = re.match(r"(\d+)-(\d+)\.jpg", fname)
                    if match:
                        person_id = int(match.group(1))
                        # 1 -> 0,..., 200 -> 199
                        if 1 <= person_id <= 200:
                            self.files.append(path)
                            self.labels.append(person_id - 1)
                        else:
                            print(f"Вне диапазона: {fname}")
                    else:
                        print(f"Не распознано: {fname}")

        if not self.files:
            raise RuntimeError(f"Не нашли .jpg в {root}/originalimages_part*/")

        print(f"Загружено {len(self.files)} изображений, {len(set(self.labels))} уникальных лиц")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

### Определение VAE

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=512):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # Encoder: 128→64→32→16→8
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),                # 128→64

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),                # 64→32

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),                # 32→16

            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),                # 16→8
        )

        self.fc_mu = nn.Linear(256 * 8 * 8, latent_dim)
        self.fc_logvar = nn.Linear(256 * 8 * 8, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 256 * 8 * 8)

        # Decoder: 8→16→32→64→128
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 4, 2, 1),  # 8→16
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 16→32
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 32→64
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # 64→128
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 3, 3, 1, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), min=-10, max=10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(h.size(0), 256, 8, 8)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), z, mu, logvar

    def sample(self, n, device):
        z = torch.randn(n, self.latent_dim).to(device)
        return self.decode(z)


class PerceptualLoss(nn.Module):
    """VGG-16 feature matching loss"""
    def __init__(self):
        super().__init__()
        from torchvision import models
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.block1 = nn.Sequential(*list(vgg)[:4]).eval()
        self.block2 = nn.Sequential(*list(vgg)[:9]).eval()
        self.block3 = nn.Sequential(*list(vgg)[:16]).eval()
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std', torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, recon, target):
        r, t = self.normalize(recon), self.normalize(target)
        return (F.mse_loss(self.block1(r), self.block1(t)) +
                F.mse_loss(self.block2(r), self.block2(t)) +
                F.mse_loss(self.block3(r), self.block3(t)))

### Подготовка данных для обучения VAE

In [ ]:
# Трансформации
transform = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

FEI_ROOT = path
train_dataset = FEIFaceDataset(root=FEI_ROOT, transform=transform)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,            
    shuffle=True,
    num_workers=2,              
    pin_memory=True,          
    persistent_workers=True   
)

### Обучение VAE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

latent_dim = 512
model = VAE(latent_dim=latent_dim)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
model.to(device)

perc_loss_fn = PerceptualLoss().to(device)

BETA_MAX = 0.0005
BETA_WARMUP = 20
LAMBDA_PERC = 0.5

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-5)

num_epochs = 80
model.train()

for epoch in range(num_epochs):
    beta = BETA_MAX * min(1.0, epoch / BETA_WARMUP)
    ep_recon, ep_kl, ep_perc, ep_total = 0, 0, 0, 0

    for img, _ in train_loader:
        img = img.to(device)
        optimizer.zero_grad()

        recon, z, mu, logvar = model(img)

        recon_loss = F.mse_loss(recon, img)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        perc_loss = perc_loss_fn(recon, img)

        loss = recon_loss + beta * kl_loss + LAMBDA_PERC * perc_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        ep_recon += recon_loss.item()
        ep_kl += kl_loss.item()
        ep_perc += perc_loss.item()
        ep_total += loss.item()

    scheduler.step()
    n = len(train_loader)

    if epoch % 5 == 0 or epoch == num_epochs - 1:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Loss: {ep_total/n:.4f} "
              f"(MSE: {ep_recon/n:.4f}, KL: {ep_kl/n:.4f}, Perc: {ep_perc/n:.4f}) "
              f"β={beta:.6f}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, 'checkpoint_vae.pth')

print("Обучение завершено")

In [ ]:
# Загрузка модели
checkpoint = torch.load('checkpoint_vae.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()


if hasattr(model, 'module'):
    model = model.module 

### Извлечение признаков

In [ ]:
def extract_latent_vae(model, data_loader, device):
    model.eval()
    latents = []
    labels = []

    with torch.no_grad():
        for img, label in data_loader:
            img = img.to(device)
            enc = model.module if hasattr(model, 'module') else model
            mu, _ = enc.encode(img)
            latents.append(mu.cpu().numpy())
            labels.append(label.numpy())

    return np.vstack(latents), np.hstack(labels)

X_latent_vae, y_labels_vae = extract_latent_vae(model, train_loader, device)

print(f"Размер латентного пространства VAE: {X_latent_vae.shape}")
print(f"Размер меток: {y_labels_vae.shape}")

### Упорядочивание признаков

In [ ]:
X_latent_ordered_fei = []

for class_id in range(200):
    indices = np.where(y_labels_vae == class_id)[0]
    X_class = X_latent_vae[indices]
    assert X_class.shape == (14, 512), f"Класс {class_id} имеет размер {X_class.shape}"
    X_latent_ordered_fei.append(X_class)

X_3d = np.stack(X_latent_ordered_fei)  # (200, 14, 512)
print(f"3D массив: {X_3d.shape}")

np.save('vae_classes_3d_fei_512.npy', X_3d)

X_2d = X_3d.reshape(-1, 512)  # (2800, 512)
np.savetxt('vae_3d_data.csv', X_2d, delimiter=',')
print(f"CSV для NCT: vae_3d_data.csv {X_2d.shape}")

torch.save({
    'model_state_dict': model.state_dict(),
    'latent_dim': 512,
    'img_size': 128,
}, 'checkpoint_vae_final.pth')
print("Чекпоинт: checkpoint_vae_final.pth")

from IPython.display import FileLink
display(FileLink('vae_classes_3d_fei_512.npy'))
display(FileLink('vae_3d_data.csv'))
display(FileLink('checkpoint_vae_final.pth'))

### Реконструкция изображений VAE

In [ ]:
enc = model.module if hasattr(model, 'module') else model
enc.eval()

# === Укажите ID людей (1-200) и номера фото (1-14) ===
SHOW_SAMPLES = [
    (1, 1), (1, 5),
    (10, 1), (10, 7),
    (50, 1), (50, 3),
    (100, 1), (100, 10),
]

# Построение индекса: person_id → [dataset_indices]
from collections import defaultdict
person_to_idx = defaultdict(list)
for i, lbl in enumerate(train_dataset.labels):
    person_to_idx[lbl].append(i)

selected_indices = []
titles = []
for person_id, photo_num in SHOW_SAMPLES:
    cls = person_id - 1
    indices = person_to_idx[cls]
    idx = indices[min(photo_num - 1, len(indices) - 1)]
    selected_indices.append(idx)
    titles.append(f'P{person_id}, ф{photo_num}')

imgs_list = []
for idx in selected_indices:
    img_tensor, _ = train_dataset[idx]
    imgs_list.append(img_tensor)
imgs = torch.stack(imgs_list).to(device)

with torch.no_grad():
    recon, _, _, _ = enc(imgs)

n_show = len(SHOW_SAMPLES)
fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))
for i in range(n_show):
    axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title(titles[i])
    axes[0, i].axis('off')

    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0).clamp(0, 1))
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('VAE: Original vs Reconstructed (128×128)', fontsize=14)
plt.tight_layout()
plt.show()

# MSE и PSNR
total_mse, total_psnr, n = 0, 0, 0
with torch.no_grad():
    for imgs_batch, _ in DataLoader(train_dataset, batch_size=256, shuffle=False):
        imgs_batch = imgs_batch.to(device)
        recon_batch, _, _, _ = enc(imgs_batch)
        mse = F.mse_loss(recon_batch, imgs_batch, reduction='none').mean(dim=[1,2,3])
        psnr = -10 * torch.log10(mse + 1e-8)
        total_mse += mse.sum().item()
        total_psnr += psnr.sum().item()
        n += imgs_batch.size(0)

print(f'\nСредний MSE: {total_mse/n:.6f}')
print(f'Средний PSNR: {total_psnr/n:.2f} dB')

In [ ]:
# Внутриклассовая vs межклассовая дисперсия (для сравнения с другими методами)
intra_dists = []
for cls in range(200):
    X_cls = X_latent_vae[y_labels_vae == cls]
    center = X_cls.mean(axis=0)
    dists = np.linalg.norm(X_cls - center, axis=1)
    intra_dists.extend(dists)

centers = np.array([X_latent_vae[y_labels_vae == c].mean(axis=0) for c in range(200)])
inter_dists = []
for i in range(200):
    for j in range(i+1, 200):
        inter_dists.append(np.linalg.norm(centers[i] - centers[j]))

intra_mean = np.mean(intra_dists)
inter_mean = np.mean(inter_dists)
ratio = inter_mean / intra_mean

print(f'Внутриклассовое расстояние (среднее): {intra_mean:.4f}')
print(f'Межклассовое расстояние (среднее): {inter_mean:.4f}')
print(f'Ratio (inter/intra): {ratio:.2f}')

# CVAE (Conditional VAE)

### Определение класса FEIFaceDatasetForCVAE

In [ ]:
class FEIFaceDatasetForCVAE(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        parts = [os.path.join(root, f"originalimages_part{i}") for i in range(1, 5)]
        self.files = []
        self.labels = [] 

        for p in parts:
            if os.path.isdir(p):
                jpg_files = sorted(glob.glob(os.path.join(p, "*.jpg")))
                for path in jpg_files:
                    filename = os.path.basename(path)
                    try:
                        person_id_str = filename.split('-')[0]  
                        person_id = int(person_id_str)
                        self.files.append(path)
                        self.labels.append(person_id - 1) 
                    except:
                        print(f"Не удалось извлечь ID из {filename}")
                        continue

        if len(self.files) == 0:
            raise RuntimeError(f"Не найдено .jpg в {root}/originalimages_part*/")

    def __len__(self):
            return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

### Подготовка данных перед обучением CVAE

In [ ]:
# Трансформации
transform = transforms.Compose([
    transforms.CenterCrop(480),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

FEI_ROOT = path
train_dataset = FEIFaceDatasetForCVAE(root=FEI_ROOT, transform=transform)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=256,             
    shuffle=True,
    num_workers=2,             
    pin_memory=True,            
    persistent_workers=True    
)

### Извлечение признаков

In [ ]:

def extract_latent_cvae(model, data_loader, device):
    model.eval()
    latents = []
    labels = []
    with torch.no_grad():
        for img, label in data_loader:
            img = img.to(device)
            label = label.to(device)
            # если torch.nn.DataParallel
            # mu, _ = model.module.encode(img, label)
            mu, _ = model.encode(img, label)
            latents.append(mu.cpu().numpy())
            labels.append(label.cpu().numpy())
    return np.vstack(latents), np.hstack(labels)


X_latent_cvae, y_labels_cvae = extract_latent_cvae(model, train_loader, device)
print(f"Latent space: {X_latent_cvae.shape}, Labels: {y_labels_cvae.shape}")

### Упорядочивание по классам полученной выборки 

In [ ]:
X_latent_ordered_fei = []

for class_id in range(200): 
    indices = np.where(y_labels_cvae == class_id)[0]
    X_class = X_latent_cvae[indices]  # (14, 256)
    X_latent_ordered_fei.append(X_class)

print(f"Количество классов: {len(X_latent_ordered_fei)}")  # 200
print(f"Размер первого класса: {X_latent_ordered_fei[0].shape}")  # (14, 256)

X_3d = np.stack(X_latent_ordered_fei)  # (200, 14, 512)


np.save('cvae_classes_3d_fei_512.npy', X_3d)
print(f"Сохранено: {X_3d.shape}")

from IPython.display import FileLink
FileLink('cvae_classes_3d_fei_512.npy')

### График t-SNE для CVAE

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns

tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)
X_tsne = tsne.fit_transform(X_latent_cvae[:1000])  # часть данных
y_sample = y_labels[:1000]

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_sample, palette='tab20', s=60)
plt.title("t-SNE of CVAE + ArcFace Latent Space")
plt.legend().set_visible(False)
plt.show()

## Функции для анализа полученных данных (см. Раздел "Сравнение VAE/CVAE")

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np

def plot_multiple_qq_plots(X_latent, n_cols=5, n_rows=2):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 6))
    axes = axes.ravel()

    # Первые 5 
    first_indices = np.arange(5)
    # Последние 5 
    last_indices = np.arange(X_latent.shape[1] - 5, X_latent.shape[1])

    all_indices = np.concatenate([first_indices, last_indices])
    titles = [f"Feature {i}" for i in first_indices] + [f"Feature {i}" for i in last_indices]

    for idx, feature_idx in enumerate(all_indices):
        data = X_latent[:, feature_idx]
        stats.probplot(data, dist="norm", plot=axes[idx])
        axes[idx].set_title(f"QQ: {titles[idx]}", fontsize=10)
        axes[idx].set_xlabel("")
        axes[idx].set_ylabel("")
        axes[idx].grid(True, alpha=0.3)
    plt.suptitle("QQ Plots: First 5 and Last 5 Latent Features", fontsize=14)
    plt.tight_layout()
    plt.show()

plot_multiple_qq_plots(X_latent_cvae)

In [ ]:
def visualize_feature_distributions(X, feature_indices=None, feature_names=None, title="Feature Distributions"):
    if feature_indices is None:
        feature_indices = np.random.choice(X.shape[1], size=12, replace=False)

    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(feature_indices):
        plt.subplot(3, 4, i + 1)
        data = X[:, idx]

        plt.hist(data, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='k', linewidth=0.5)

        mu, std = stats.norm.fit(data)
        xmin, xmax = plt.xlim()
        x = np.linspace(xmin, xmax, 100)
        p = stats.norm.pdf(x, mu, std)
        plt.plot(x, p, 'r-', lw=2, label=f'N({mu:.2f}, {std:.2f}^2)')

        if feature_names is not None and idx < len(feature_names):
            feature_label = feature_names[idx]
        else:
            feature_label = f"Feature {idx}"

        plt.title(f"{feature_label}")
        plt.xlabel("Value")
        plt.ylabel("Density")
        plt.legend(fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

visualize_feature_distributions(X_latent, title="Distribution of 12 Random Features")

In [ ]:
def plot_correlation_matrix_subset(X, n_features=50, title="Correlation Matrix (Subset)"):
    indices = np.random.choice(X.shape[1], size=n_features, replace=False)
    X_subset = X[:, indices]
    # Корреляция Пирсона
    corr_matrix = np.corrcoef(X_subset, rowvar=False)  # [n_features, n_features]

    plt.figure(figsize=(10, 8))
    im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, interpolation='none')
    plt.colorbar(im)
    plt.title(f"Correlation Matrix of {n_features} Features")
    plt.xlabel("Feature Index")
    plt.ylabel("Feature Index")
    plt.tight_layout()
    plt.show()

    # Статистика по корреляциям
    upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
    print(f"\nСтатистика по корреляциям (выборка {n_features} признаков):")
    print(f"Средняя корреляция: {upper_tri.mean():.3f}")
    print(f"Медианная корреляция: {np.median(upper_tri):.3f}")
    print(f"Доля |r| > 0.5: {np.mean(np.abs(upper_tri) > 0.5):.3f}")
    print(f"Доля |r| > 0.7: {np.mean(np.abs(upper_tri) > 0.7):.3f}")

# plot_correlation_matrix_subset(X_latent_cvae, n_features=100)

# Сравнение VAE/CVAE

## t-SNE: как кластеризуются классы?


In [ ]:
from sklearn.manifold import TSNE
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Подвыборка для скорости
n_sample = 2000
idx = np.random.choice(2800, n_sample, replace=False)

X_vae_sample = X_latent_vae[idx]
X_cvae_sample = X_latent_cvae[idx]
y_sample = y_labels_vae[idx]

tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)

plt.figure(figsize=(14, 6))

# VAE
plt.subplot(1, 2, 1)
Z_vae = tsne.fit_transform(X_vae_sample)
sns.scatterplot(x=Z_vae[:, 0], y=Z_vae[:, 1], hue=y_sample, palette='tab20', s=50, legend=False)
plt.title("t-SNE: VAE Latent Space")
plt.axis('off')

# CVAE
plt.subplot(1, 2, 2)
Z_cvae = tsne.fit_transform(X_cvae_sample)
sns.scatterplot(x=Z_cvae[:, 0], y=Z_cvae[:, 1], hue=y_sample, palette='tab20', s=50, legend=False)
plt.title("t-SNE: CVAE Latent Space")
plt.axis('off')

plt.suptitle("Сравнение латентных пространств VAE vs CVAE (FEI Face)")
plt.tight_layout()
plt.show()

## Корреляция между признаками


In [ ]:
def plot_correlation_matrix_subset(X, n_features=50, title="Correlation Matrix (Subset)"):
    indices = np.random.choice(X.shape[1], size=n_features, replace=False)
    X_subset = X[:, indices]

    # Корреляция Пирсона
    corr_matrix = np.corrcoef(X_subset, rowvar=False)  # [n_features, n_features]

    plt.figure(figsize=(10, 8))
    im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, interpolation='none')
    plt.colorbar(im)
    plt.title(f"Correlation Matrix of {n_features} Features")
    plt.xlabel("Feature Index")
    plt.ylabel("Feature Index")
    plt.tight_layout()
    plt.show()

    # Статистика по корреляциям
    upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
    print(f"\nСтатистика по корреляциям (выборка {n_features} признаков):")
    print(f"Средняя корреляция: {upper_tri.mean():.3f}")
    print(f"Медианная корреляция: {np.median(upper_tri):.3f}")
    print(f"Доля |r| > 0.5: {np.mean(np.abs(upper_tri) > 0.5):.3f}")
    print(f"Доля |r| > 0.7: {np.mean(np.abs(upper_tri) > 0.7):.3f}")

# plot_correlation_matrix_subset(X_latent_cvae, n_features=100)

In [ ]:
# VAE
plot_correlation_matrix_subset(
    X_latent_vae,
    n_features=100,
    title="VAE: Correlation Matrix (100 признаков)"
)

# CVAE
plot_correlation_matrix_subset(
    X_latent_cvae,
    n_features=100,
    title="CVAE: Correlation Matrix (100 признаков)"
)

## Внутри- и межклассовая дисперсия

In [ ]:
from sklearn.metrics import pairwise_distances

def intra_inter_class_variance(X, y):
    classes = np.unique(y)
    intra_dists = []
    inter_dists = []

    # Внутриклассовые расстояния
    for cls in classes:
        X_cls = X[y == cls]
        if len(X_cls) < 2:
            continue
        dists = pairwise_distances(X_cls)
        upper_tri = dists[np.triu_indices_from(dists, k=1)]
        intra_dists.extend(upper_tri)

    # Межклассовые 
    class_pairs = [(i, j) for i in classes for j in classes if i < j]
    sampled_pairs = np.random.choice(len(class_pairs), size=min(100, len(class_pairs)), replace=False)

    for idx in sampled_pairs:
        i, j = class_pairs[idx]
        X_i = X[y == i]
        X_j = X[y == j]
        X_i = X_i[np.random.choice(len(X_i), size=min(10, len(X_i)), replace=False)]
        X_j = X_j[np.random.choice(len(X_j), size=min(10, len(X_j)), replace=False)]
        cross_dists = pairwise_distances(X_i, X_j).flatten()
        inter_dists.extend(cross_dists)

    intra_mean = np.mean(intra_dists) if intra_dists else 0AA
    inter_mean = np.mean(inter_dists) if inter_dists else 0
    ratio = inter_mean / intra_mean if intra_mean > 0 else np.inf

    return intra_mean, inter_mean, ratio

# Сравнение
intra_vae, inter_vae, ratio_vae = intra_inter_class_variance(X_latent_vae, y_labels_vae)
intra_cvae, inter_cvae, ratio_cvae = intra_inter_class_variance(X_latent_cvae, y_labels_cvae)

print(f"VAE:  Intra={intra_vae:.3f}, Inter={inter_vae:.3f}, Ratio={ratio_vae:.3f}")
print(f"CVAE: Intra={intra_cvae:.3f}, Inter={inter_cvae:.3f}, Ratio={ratio_cvae:.3f}")

## Качество реконструкции

In [ ]:
checkpoint = torch.load('checkpoint_vae.pth', map_location=device)
vae_model = VAE(latent_dim=512) 

if torch.cuda.device_count() > 1:
    vae_model = torch.nn.DataParallel(vae_model)
vae_model.to(device)


vae_model.load_state_dict(checkpoint['model_state_dict'])

if hasattr(vae_model, 'module'):
    vae_model = vae_model.module

vae_model.eval()

print("VAE модель загружена в переменную `vae_model`.")

In [ ]:
cvae_path = 'cvae_fei_face_200classes.pth'
cvae_model = CVAE_ArcFace(latent_dim=512, num_classes=200)  

cvae_model.to(device)
cvae_model.load_state_dict(torch.load(cvae_path, map_location=device))

if hasattr(cvae_model, 'module'):
    cvae_model = cvae_model.module

cvae_model.eval()

print("CVAE модель загружена в переменную `cvae_model`.")

In [ ]:
def show_reconstructions_vae(model, data_loader, device, n=6, title="Reconstructions"):
    model.eval()
    with torch.no_grad():
        imgs, _ = next(iter(data_loader))
        imgs = imgs[:n].to(device)
        recons, _, _, _ = model(imgs)

        fig, axes = plt.subplots(2, n, figsize=(12, 5))
        for i in range(n):
            axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")

            axes[1, i].imshow(recons[i].cpu().permute(1, 2, 0))
            axes[1, i].set_title("Reconstructed")
            axes[1, i].axis("off")
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()

show_reconstructions_vae(vae_model, train_loader, device, title="VAE Reconstructions")

In [ ]:
def show_reconstructions_cvae(model, data_loader, device, n=6, title="Reconstructions"):
    model.eval()
    with torch.no_grad():
        data = next(iter(data_loader))
        imgs, labels = data[0][:n].to(device), data[1][:n].to(device)  # берем и изображения, и метки
        
        recons, _, _, _, _ = model(imgs, labels)

        fig, axes = plt.subplots(2, n, figsize=(12, 5))
        for i in range(n):
            axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")

            axes[1, i].imshow(recons[i].cpu().permute(1, 2, 0))
            axes[1, i].set_title("Reconstructed")
            axes[1, i].axis("off")
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()

# Для CVAE
show_reconstructions_cvae(cvae_model, train_loader, device, title="CVAE Reconstructions")

In [ ]:
import matplotlib.pyplot as plt

def compare_reconstructions(vae_model, cvae_model, data_loader, device, n=6, title="VAE vs CVAE Reconstructions"):
    vae_model.eval()
    cvae_model.eval()
    
    with torch.no_grad():
        data = next(iter(data_loader))
        imgs, labels = data[0][:n].to(device), data[1][:n].to(device)
        
        # VAE: только изображение
        recon_vae, _, _, _ = vae_model(imgs)
        
        # CVAE: изображение и метка
        recon_cvae, _, _, _, _ = cvae_model(imgs, labels)
        
        fig, axes = plt.subplots(3, n, figsize=(14, 7))
        
        for i in range(n):
            # Оригинал
            axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")
            
            # VAE
            axes[1, i].imshow(recon_vae[i].cpu().permute(1, 2, 0))
            axes[1, i].set_title("VAE")
            axes[1, i].axis("off")
            
            # CVAE
            axes[2, i].imshow(recon_cvae[i].cpu().permute(1, 2, 0))
            axes[2, i].set_title("CVAE")
            axes[2, i].axis("off")
        
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()


compare_reconstructions(
    vae_model=vae_model,
    cvae_model=cvae_model,
    data_loader=train_loader,
    device=device,
    n=6,
    title="Сравнение VAE и CVAE на FEI Face"
)

# Продолжение экспериментов с CIFAR-100

### Подготовка данных

In [ ]:
import kagglehub

path = kagglehub.dataset_download("fedesoriano/cifar100")

print("Path to dataset files:", path)

In [ ]:
import os
import shutil

input_dir = '/kaggle/input/cifar100'
data_dir = './data/cifar-100-python' 

os.makedirs(data_dir, exist_ok=True)

for fname in ['train', 'test', 'meta']:
    src = os.path.join(input_dir, fname)
    dst = os.path.join(data_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Скопировано: {src} → {dst}")
    else:
        print(f"Файл не найден: {src}")

print("Готово! Теперь можно загружать CIFAR-100.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

# Трансформации
transform = transforms.Compose([
    transforms.ToTensor(),  # [0,1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # [-1,1]
])

train_dataset = datasets.CIFAR100(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR100(
    root='./data',
    train=False,
    download=False,
    transform=transform
)

print(f"Train size: {len(train_dataset)}")  
print(f"Test size: {len(test_dataset)}")   
       

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

### Упрощённый CVAE для 32×32

In [ ]:
class CVAE_CIFAR(nn.Module):
    def __init__(self, latent_dim=256, num_classes=100):
        super(CVAE_CIFAR, self).__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.label_emb = nn.Embedding(num_classes, 32)  # меньше, чем в FEI

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3 + 32, 32, 3, stride=1, padding=1),  # 32×32
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),     # 32 → 16
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),    # 16 → 8
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),   # 8 → 4
            nn.ReLU(),  # [B, 256, 4, 4]
        )

        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim + 32, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),  # 4 → 8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),   # 8 → 16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),    # 16 → 32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=1, padding=1),
            nn.Tanh()  # выход в [-1,1] — из-за Normalize
        )

        # ArcFace головка
        # self.arcface = ArcMarginProduct(in_features=latent_dim, out_features=num_classes)

    def encode(self, x, y):
        B = x.size(0)
        label_emb = self.label_emb(y).unsqueeze(-1).unsqueeze(-1)  # (B, 32, 1, 1)
        label_emb = label_emb.expand(-1, -1, 32, 32)
        x_cond = torch.cat([x, label_emb], dim=1)  # (B, 35, 32, 32)
        h = self.encoder(x_cond)  # (B, 256, 4, 4)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        label_emb = self.label_emb(y)
        z_cond = torch.cat([z, label_emb], dim=1)
        h = self.fc_decode(z_cond)
        h = h.view(h.size(0), 256, 4, 4)
        return self.decoder(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, y)
        arc_logits = self.arcface(mu, y)
        return x_recon, z, mu, logvar, arc_logits

### Обучение CVAE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

model = CVAE_CIFAR(latent_dim=512, num_classes=100).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def cvae_loss(recon_x, x, mu, logvar, arc_logits, labels, beta=0.001, lambda_arc=0.2):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    ce_loss = F.cross_entropy(arc_logits, labels)
    return recon_loss + beta * kl_loss + lambda_arc * ce_loss

# Обучение
model.train()
for epoch in range(50):
    for img, label in train_loader:
        img = img.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        recon, z, mu, logvar, arc_logits = model(img, label)
        # ---------------- DEBUG -------------------
        beta = 0.001
        lambda_arc = min(1.0, 0.1 + 0.9 * (epoch / 50))  # растёт от 0.1 до 1.0
        
        recon_loss = F.mse_loss(recon, img, reduction='mean')
        logvar_clipped = torch.clamp(logvar, -10, 10)
        kl_loss = -0.5 * torch.mean(1 + logvar_clipped - mu.pow(2) - torch.exp(logvar_clipped))
        ce_loss = F.cross_entropy(arc_logits, label)
        loss = recon_loss + beta * kl_loss + lambda_arc * ce_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    if epoch % 10 == 0 or epoch == 49:
        print(f"Epoch [{epoch+1}/50] "
              f"Loss: {loss.item():.4f} "
              f"(Recon: {recon_loss.item():.4f}, "
              f"KL: {kl_loss.item():.4f}, "
              f"Arc: {ce_loss.item():.4f})")
        # ---------------- DEBUG -------------------
        # loss = cvae_loss(recon, img, mu, logvar, arc_logits, label)
    # if epoch % 10 == 0:
    #     print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}")

torch.save(model.state_dict(), 'cvae_cifar100_model.pth')

### Визуализация

In [ ]:
def show_recs(model, loader, device, n=6):
    model.eval()
    with torch.no_grad():
        data = next(iter(loader))
        imgs, labels = data[0][:n].to(device), data[1][:n].to(device)
        recon, _, _, _, _ = model(imgs, labels)
        recon = (recon + 1) / 2  # из [-1,1] в [0,1]
        imgs = (imgs + 1) / 2

        fig, axes = plt.subplots(2, n, figsize=(12, 5))
        for i in range(n):
            axes[0, i].imshow(imgs[i].cpu().permute(1, 2, 0))
            axes[0, i].axis("off")
            axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0))
            axes[1, i].axis("off")
        plt.suptitle("CVAE on CIFAR-100: Reconstructions")
        plt.tight_layout()
        plt.show()

show_recs(model, test_loader, device)

### Извлечение латентных векторов

In [ ]:
def extract_latent(model, loader, device):
    model.eval()
    latents = []
    labels = []
    with torch.no_grad():
        for img, label in loader:
            img = img.to(device)
            label = label.to(device)
            mu, _ = model.encode(img, label)
            latents.append(mu.cpu().numpy())
            labels.append(label.cpu().numpy())
    return np.vstack(latents), np.hstack(labels)

X_latent, y_labels = extract_latent(model, test_loader, device)
print(f"Размер латентного пространства: {X_latent.shape}") 

In [ ]:
X_latent_ordered = []
y_ordered = []

for class_id in range(100):
    indices = np.where(y_labels == class_id)[0]
    X_class = X_latent[indices]  # (100, 512)
    X_latent_ordered.append(X_class)


In [ ]:
sample_labels = []
for class_id in range(100):
    indices = np.where(y_labels == class_id)[0]
    sample_idx = indices[0]
    sample_labels.append(y_labels[sample_idx])

In [ ]:
classes_3d_cifar = [
    [vec.tolist() for vec in X_latent_ordered[i]]
    for i in range(100)
]
print(f"CIFAR: {len(classes_3d_cifar)} классов × {len(classes_3d_cifar[0])} изображений × {len(classes_3d_cifar[0][0])} признаков")

X_3d = np.array(classes_3d_cifar) 

np.save('classes_3d_cifar_512.npy', X_3d)
print("Сохранено: classes_3d_cifar_512.npy")

from IPython.display import FileLink
FileLink('classes_3d_cifar_512.npy')

### Анализ латентных векторов

In [ ]:
# t-SNE
from sklearn.manifold import TSNE
import seaborn as sns

tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42)
X_sample = X_latent[:1000]
y_sample = y_labels[:1000]
X_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y_sample, palette='tab10', s=60)
plt.title("t-SNE of CVAE Latent Space (CIFAR-100)")
plt.legend().set_visible(False)
plt.axis('off')
plt.show()

# k-NN accuracy
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_latent[:5000], y_labels[:5000])
acc = knn.score(X_latent[5000:], y_labels[5000:])
print(f"1-NN Accuracy: {acc:.4f}")